## Prescription Parser Agent

In [45]:
import uuid
import base64
from flask import Flask, request, jsonify
from dotenv import load_dotenv, find_dotenv

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver  
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List
from datetime import datetime as dt

load_dotenv(find_dotenv(), override=True)

True

In [46]:
class PrescInfo(BaseModel):
    date: str = Field(description="Date formatted in MM-DD-YYYY")
    prescription_name: str = Field(description="Name of prescription given exactly.")
    active_ingrediants: List[str] = Field(description="list of active ingrediants in drug")
    patient_first_name: str = Field(description="First name of patient")
    patient_last_name: str = Field(description="Last name of patient")
    patient_DOB: str = Field(description="Date of birth of patient in MM/DD/YYYY")
    medication_details: str = Field(description="Details and instructions on usage")
    num_refills: float = Field(description="Maximum number of refills allowed")
    is_signed: bool = Field(description="True if the description is valid and signed, false otherwise")
    fulltext: str = Field(description="All text on prescription for analysis")
    

In [47]:
with open('./data/proair_presc.png', 'rb') as imgfp:
    img_str = base64.b64encode(imgfp.read()).decode("utf-8")

In [48]:
presc_processor_prompt = SystemMessage(content=("You are an expert prescription parser.\n" 
    "Your job is to parse the prescription into its components for further analysis."))

presc_human_data = HumanMessage(content=[
    {"type": "text", "text": "Parse the following prescription into the output format"},
    {
        "type": "image_url",
        "image_url": {"url":f"data:image/jpeg;base64,{img_str}"}
    }
])

messages = [
    presc_processor_prompt, presc_human_data
]


In [49]:
llm = ChatOpenAI(temperature=0.2, model='gpt-5.4')
threadID = f"presc-{uuid.uuid4().hex[:8]}"
mem = InMemorySaver()
mem.delete_thread(threadID)
presc_agent = create_agent(
    model=llm, 
    response_format=PrescInfo,
    checkpointer=mem
)

In [50]:
res = presc_agent.invoke({"messages": messages}, config={"configurable": {"thread_id": threadID}})

In [51]:
receipt_info: PrescInfo = res["structured_response"]
print(receipt_info)

date='05-24-2024' prescription_name='ProAir HFA (albuterol sulfate)' active_ingrediants=['albuterol sulfate'] patient_first_name='Emily' patient_last_name='Harris' patient_DOB='07/12/1998' medication_details='90 mcg inhalation aerosol. Inhale 2 puffs by mouth every 4 to 6 hours as needed for wheezing or shortness of breath. Dispense: 1 inhaler.' num_refills=2.0 is_signed=True fulltext='Michael J. Thompson, MD\nINTERNAL MEDICINE\n1234 Oakview Drive, Suite 200\nRaleigh, NC 27609\n(919) 555-2678\nNPI: 1234567890\nDate: 5/24/24\nPatient Name: Emily J. Harris\nDOB: 7/12/1998\nRx\nProAir HFA (albuterol sulfate)\n90 mcg inhalation aerosol\nInhale 2 puffs by mouth every 4 to 6 hours as needed for wheezing or shortness of breath.\nDispense: 1 inhaler\nRefills: 2\nSignature'


### Prescription 